# Kalpa Plus membership investigation
All data is synthetic. Follow DATA_DICTIONARY.md. Set DATA_DIR to your downloaded folder. You can load the CSVs from local storage or mounted Google Drive.

In [2]:
from pathlib import Path
import pandas as pd

In [3]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_DIR = Path(".")  # Set this to the folder containing the three CSV files.
AS_OF = pd.Timestamp("2026-10-09 23:59:59")

members = pd.read_csv(
    DATA_DIR / "members.csv",
    dtype={"member_id": "string"},
    parse_dates=["join_date"],
)
renewals = pd.read_csv(
    DATA_DIR / "renewal_opportunities.csv",
    dtype={"renewal_id": "string", "member_id": "string"},
    parse_dates=["expiry_date", "renewal_deadline"],
)
events = pd.read_csv(
    DATA_DIR / "member_events.csv",
    dtype={"event_id": "string", "renewal_id": "string", "member_id": "string"},
    parse_dates=["event_time"],
)

print("Rows (members, renewal opportunities, events):", members.shape, renewals.shape, events.shape)


Rows (members, renewal opportunities, events): (8580, 8) (11580, 7) (152057, 10)


## Before coding
What is one row in each table? Which key is unique? Which join can multiply rows? State the denominator for renewal rate. Why does an October expiry need special treatment?

## Answers before coding

- `members`: one row per member; unique key `member_id`.
- `renewal_opportunities`: one row per paid-term expiry/renewal chance; unique key `renewal_id`.
- `member_events`: one row per event; unique key `event_id`. A single opportunity can have many events.
- Join `members.member_id` to `renewal_opportunities.member_id` (one-to-many); join opportunities to events by `renewal_id` (one-to-many). Joining raw events directly can multiply opportunity rows, so summarize events to one row per `renewal_id` before joining for renewal-rate work.
- Renewal-rate denominator: mature renewal opportunities whose complete window has closed. A member who has not reached the first expiry has no opportunity row and is not yet eligible.
- A renewal window is expiry minus 7 days through deadline (expiry plus 7 days), inclusive. An October opportunity with a deadline after the 9 October snapshot is still unresolved, not a non-renewal.


In [6]:
display(members.head())


  member_id  join_date      plan_code  annual_fee acquisition_channel       city status_2025_10_09 status_2026_10_09
0  MP000467 2023-09-22  PLUS_STANDARD        1499               STORE   Gurugram              PAID              PAID
1  MP005549 2025-09-04   PLUS_PREMIUM        2499                 WEB  Bengaluru              PAID              PAID
2  MP000708 2023-09-13  PLUS_STANDARD        1499               STORE  Bengaluru              PAID              PAID
3  MP006266 2025-10-04  PLUS_STANDARD        1499                 APP      Delhi              PAID              PAID
4  MP000081 2023-09-26  PLUS_STANDARD        1499                 APP     Jaipur              PAID              PAID


In [7]:
display(renewals.head())


  renewal_id member_id  renewal_number expiry_date renewal_deadline  renewal_fee  offered_discount
0   RN004822  MP002964               1  2025-09-02       2025-09-09         2499                 0
1   RN007937  MP001571               3  2026-09-14       2026-09-21         2499                 0
2   RN002399  MP002040               1  2024-09-05       2024-09-12         1499                 0
3   RN009951  MP004511               1  2026-09-18       2026-09-25         1499                 0
4   RN006488  MP003536               1  2025-09-13       2025-09-20         1499                 0


In [8]:
display(events.head())


    event_id member_id renewal_id          event_time    event_type channel  outcome       detail_code  amount  event_cost
0  EV0023859  MP000456   RN001802 2024-07-03 08:50:00  BENEFIT_USED     APP  SUCCESS      MEMBER_OFFER       0          10
1  EV0005255  MP000612   RN000390 2024-07-03 11:34:00  BENEFIT_USED     APP  SUCCESS     FREE_DELIVERY       0          27
2  EV0000747  MP001531   RN000060 2024-07-03 13:20:00  BENEFIT_USED     WEB  SUCCESS     FREE_DELIVERY       0          55
3  EV0029706  MP002497   RN002203 2024-07-03 17:39:00  BENEFIT_USED     APP  SUCCESS     FREE_DELIVERY       0          53
4  EV0031926  MP001821   RN002456 2024-07-03 17:47:00  BENEFIT_USED     APP  SUCCESS  PRIORITY_SUPPORT       0          13


## Outcome
Compare paid membership at both snapshots. Define a renewal metric and business guardrail.

### Findings and definitions

Paid membership increased from **5,080** at the 9 October 2025 snapshot to **6,024** at the 9 October 2026 snapshot: a net increase of **944 (18.6%)**. Of the 2025 paid members, 3,824 remained paid and 1,256 were expired in 2026; 2,200 members marked `NOT_JOINED` in 2025 were paid in 2026. The change combines new members and expirations, so it is not a renewal rate.

Define renewal rate as **successful renewal opportunities ÷ mature renewal opportunities**. Count an opportunity as successful if it has at least one `PAYMENT_ATTEMPT` with `outcome == "SUCCESS"` from seven days before expiry through the seven-day deadline, inclusive. A mature opportunity has a deadline on or before 9 October 2026. October opportunities with later deadlines remain open and are excluded. Each opportunity counts once, regardless of how many event rows it has.

Business guardrail: only scale an intervention if its cost per additional renewal is no greater than the expected contribution margin from that renewal, and customer/billing quality measures do not worsen. These files do not contain full contribution margin, so the affordable cost cap must be supplied by Finance before a pilot.


In [11]:
# Paid members at both snapshots and member-level status transitions
paid_2025 = members["status_2025_10_09"].eq("PAID").sum()
paid_2026 = members["status_2026_10_09"].eq("PAID").sum()
print(f"Paid on 2025-10-09: {paid_2025:,}")
print(f"Paid on 2026-10-09: {paid_2026:,}")
print(f"Net change: {paid_2026 - paid_2025:+,} ({paid_2026 / paid_2025 - 1:.1%})")
display(pd.crosstab(members["status_2025_10_09"], members["status_2026_10_09"]))

# One row per event joined to the opportunity it belongs to.
event_opportunities = events.merge(
    renewals[["renewal_id", "member_id", "expiry_date", "renewal_deadline"]],
    on="renewal_id", how="left", validate="many_to_one",
    suffixes=("_event", "_opportunity"),
)
assert event_opportunities["expiry_date"].notna().all()
assert event_opportunities["member_id_event"].eq(
    event_opportunities["member_id_opportunity"]
).all()

event_day = event_opportunities["event_time"].dt.normalize()
in_renewal_window = (
    (event_day >= event_opportunities["expiry_date"] - pd.Timedelta(days=7))
    & (event_day <= event_opportunities["renewal_deadline"])
)
payment_events = event_opportunities[
    in_renewal_window & event_opportunities["event_type"].eq("PAYMENT_ATTEMPT")
].copy()
payment_events["successful_payment"] = payment_events["outcome"].eq("SUCCESS")
payment_summary = payment_events.groupby("renewal_id").agg(
    payment_attempts=("event_id", "size"),
    renewed=("successful_payment", "any"),
)

# Preserve every opportunity, including those with no payment event.
opportunities = renewals.merge(payment_summary, on="renewal_id", how="left")
opportunities = opportunities.merge(
    members[["member_id", "join_date", "plan_code", "annual_fee", "acquisition_channel", "city"]],
    on="member_id", how="left", validate="many_to_one",
)
opportunities["payment_attempts"] = opportunities["payment_attempts"].fillna(0).astype(int)
opportunities["renewed"] = opportunities["renewed"].fillna(False).astype(bool)
opportunities["mature"] = opportunities["renewal_deadline"].dt.normalize() <= AS_OF.normalize()
mature_opportunities = opportunities[opportunities["mature"]].copy()

print("Mature opportunities:", len(mature_opportunities))
print("Open/unresolved opportunities:", int((~opportunities["mature"]).sum()))
print("Renewals among mature opportunities:", int(mature_opportunities["renewed"].sum()))
print("Overall mature renewal rate:", f"{mature_opportunities['renewed'].mean():.1%}")


Paid on 2025-10-09: 5,080
Paid on 2026-10-09: 6,024
Net change: +944 (18.6%)
status_2026_10_09  EXPIRED  PAID
status_2025_10_09               
EXPIRED               1300     0
NOT_JOINED               0  2200
PAID                  1256  3824
Mature opportunities: 10980
Open/unresolved opportunities: 600
Renewals among mature opportunities: 8784
Overall mature renewal rate: 80.0%


## Deconstruction
Build one row per renewal opportunity. Compare mature September opportunities and first versus later renewals. What does a company-wide average hide?

### Findings

The source grains are: one row per member in `members`, one row per expiry opportunity in `renewal_opportunities`, and one row per recorded event in `member_events`. `member_id` links members to many opportunities; `renewal_id` links an opportunity to many events. Aggregate events to `renewal_id` before joining them to opportunities, or event rows will multiply the opportunity counts.

There are **8,580 members**, **11,580 opportunities**, and **152,057 events**, with unique primary keys. A member not yet at first expiry has no first-renewal opportunity and is not in the renewal denominator. There are 2,200 such not-yet-due members. There are 600 October opportunities with open renewal windows; exclude them from rates.

For September, the overall mature renewal rate was **80% in both years**. First renewals fell from **70% (1,400 / 2,000)** to **55% (704 / 1,280)**, while later renewals stayed at **90%**. First renewals made up 50% of September 2025 opportunities but only 28.6% in September 2026, so the company-wide average hides the first-renewal decline.

First-renewal rates by joining cohort were 80% for September 2023 joiners (their September 2024 expiry), 70% for September 2024 joiners, and 55% for September 2025 joiners. Compare only cohorts whose observation windows are complete.


In [14]:
# One row per renewal opportunity is already in `opportunities`.
# Verify primary keys, foreign keys, and snapshot cutoffs.
print("Unique keys:", members["member_id"].nunique(), renewals["renewal_id"].nunique(), events["event_id"].nunique())
print("Renewal member keys found:", renewals["member_id"].isin(members["member_id"]).all())
print("Event renewal keys found:", events["renewal_id"].isin(renewals["renewal_id"]).all())
print("Expiry dates:", renewals["expiry_date"].min(), "to", renewals["expiry_date"].max())

opportunities["period"] = opportunities["expiry_date"].dt.to_period("M").astype(str)
opportunities["stage"] = np.where(opportunities["renewal_number"].eq(1), "First", "Later")
september = mature_opportunities[
    mature_opportunities["expiry_date"].dt.to_period("M").astype(str).isin(["2025-09", "2026-09"])
].copy()
september["period"] = september["expiry_date"].dt.to_period("M").astype(str)
september["stage"] = np.where(september["renewal_number"].eq(1), "First", "Later")

by_stage = september.groupby(["period", "stage"]).agg(
    opportunities=("renewal_id", "nunique"), renewals=("renewed", "sum")
).reset_index()
by_stage["renewal_rate"] = by_stage["renewals"] / by_stage["opportunities"]
display(by_stage)

by_period = september.groupby("period").agg(
    opportunities=("renewal_id", "nunique"), renewals=("renewed", "sum")
).reset_index()
by_period["renewal_rate"] = by_period["renewals"] / by_period["opportunities"]
display(by_period)

# First-renewal join cohorts; each complete first opportunity appears once.
first_cohorts = mature_opportunities[mature_opportunities["renewal_number"].eq(1)].copy()
first_cohorts["join_cohort"] = first_cohorts["join_date"].dt.to_period("M").astype(str)
cohort_summary = first_cohorts.groupby("join_cohort").agg(
    opportunities=("renewal_id", "nunique"), renewals=("renewed", "sum")
).reset_index()
cohort_summary["renewal_rate"] = cohort_summary["renewals"] / cohort_summary["opportunities"]
display(cohort_summary)


Unique keys: 8580 11580 152057
Renewal member keys found: True
Event renewal keys found: True
Expiry dates: 2024-09-01 00:00:00 to 2026-10-09 00:00:00
    period  stage  opportunities  renewals  renewal_rate
0  2025-09  First           2000      1400          0.70
1  2025-09  Later           2000      1800          0.90
2  2026-09  First           1280       704          0.55
3  2026-09  Later           3200      2880          0.90
    period  opportunities  renewals  renewal_rate
0  2025-09           4000      3200           0.8
1  2026-09           4480      3584           0.8
  join_cohort  opportunities  renewals  renewal_rate
0     2023-09           2500      2000          0.80
1     2024-09           2000      1400          0.70
2     2025-09           1280       704          0.55


## Transitions
Count benefits in each of the two complete 30-day windows. Build a usage-state transition table. Does inactive usage mean expired membership?

### Findings

For each opportunity, benefit use is counted in the 60–31 days before expiry and in the 30–1 days before expiry. The provided event extract covers these complete pre-expiry windows. Usage states are exercise definitions: **Inactive = 0**, **Occasional = 1–3**, and **Active = 4+** benefit uses.

The transition table shows people moving both up and down across the two windows. In the member’s latest recorded opportunity, 323 of 1,088 people with inactive recent use were still marked `PAID` at the 2026 snapshot (29.7%). Conversely, 917 of 3,490 people with active recent use were marked `EXPIRED` (26.3%). Inactive usage therefore does not mean expired membership; engagement and billing status are separate.


In [17]:
benefit_events = event_opportunities[
    event_opportunities["event_type"].eq("BENEFIT_USED")
].copy()
benefit_events["days_before_expiry"] = (
    benefit_events["expiry_date"].dt.normalize()
    - benefit_events["event_time"].dt.normalize()
).dt.days

benefit_counts = benefit_events.groupby("renewal_id").agg(
    prior_30=("days_before_expiry", lambda d: d.between(31, 60).sum()),
    recent_30=("days_before_expiry", lambda d: d.between(1, 30).sum()),
)
usage = opportunities[["renewal_id", "member_id", "expiry_date"]].merge(
    benefit_counts, on="renewal_id", how="left", validate="one_to_one"
)
usage[["prior_30", "recent_30"]] = usage[["prior_30", "recent_30"]].fillna(0).astype(int)

def usage_state(n):
    if n == 0:
        return "Inactive"
    if n < 4:
        return "Occasional"
    return "Active"

usage["prior_state"] = usage["prior_30"].map(usage_state)
usage["recent_state"] = usage["recent_30"].map(usage_state)
display(pd.crosstab(usage["prior_state"], usage["recent_state"], margins=True))

# Compare the latest opportunity per member with snapshot billing status.
latest_usage = (
    usage.sort_values("expiry_date")
    .drop_duplicates("member_id", keep="last")
    .merge(members[["member_id", "status_2026_10_09"]], on="member_id", validate="one_to_one")
)
display(pd.crosstab(latest_usage["recent_state"], latest_usage["status_2026_10_09"]))
display(pd.crosstab(
    latest_usage["recent_state"], latest_usage["status_2026_10_09"], normalize="index"
).round(3))


recent_state  Active  Inactive  Occasional    All
prior_state                                      
Active          4966      1251        2176   8393
Inactive         499        86         219    804
Occasional      1515       183         685   2383
All             6980      1520        3080  11580
status_2026_10_09  EXPIRED  PAID
recent_state                    
Active                 917  2573
Inactive               765   323
Occasional             874   928
status_2026_10_09  EXPIRED   PAID
recent_state                     
Active               0.263  0.737
Inactive             0.703  0.297
Occasional           0.485  0.515


## Alternatives
Separate non-renewals with failed payment attempts from those with no recorded attempt. Examine usage, failure codes and support contacts. Which explanation survives?

### Findings

Among **1,676 mature first-renewal non-renewals**, 540 had payment attempts but no successful payment, while 1,136 had no recorded payment attempt. In the failed-attempt group, mean benefit use stayed high (8.4 in each 30-day window), and 81.7% had a support contact. The 1,136 with no attempt averaged 6.2 uses in the prior window but only 1.1 in the recent window; 32.0% had support contact. Their logged support topics included benefits questions, renewal help, and some price queries.

Failed attempts were dominated by `PSP_TIMEOUT` (1,144 event rows among these first-renewal non-renewals), followed by `AUTHENTICATION_FAILED` (327) and `INSUFFICIENT_FUNDS` (149). This supports investigating payment recovery for those who attempted. For no-attempt cases, the usage drop and support topics make declining perceived value or renewal friction plausible hypotheses, but they do not establish why anyone left. A failure code does not prove a software root cause, and association is not causation.


In [20]:
# Mature first-renewal non-renewals only.
first_nonrenewals = mature_opportunities[
    mature_opportunities["renewal_number"].eq(1) & ~mature_opportunities["renewed"]
].copy()
first_nonrenewals["attempt_group"] = np.where(
    first_nonrenewals["payment_attempts"].gt(0),
    "Attempted, no success", "No recorded attempt"
)

support_counts = event_opportunities[
    event_opportunities["event_type"].eq("SUPPORT_CONTACT")
].groupby("renewal_id").agg(support_contacts=("event_id", "size"))
first_nonrenewals = first_nonrenewals.merge(
    usage[["renewal_id", "prior_30", "recent_30"]], on="renewal_id", validate="one_to_one"
).merge(support_counts, on="renewal_id", how="left", validate="one_to_one")
first_nonrenewals["support_contacts"] = first_nonrenewals["support_contacts"].fillna(0)

group_summary = first_nonrenewals.groupby("attempt_group").agg(
    opportunities=("renewal_id", "nunique"),
    mean_prior_uses=("prior_30", "mean"),
    mean_recent_uses=("recent_30", "mean"),
    support_contact_share=("support_contacts", lambda x: x.gt(0).mean()),
)
display(group_summary.round(3))

nonrenewal_ids = first_nonrenewals[["renewal_id", "attempt_group"]]
failed_first = payment_events[
    payment_events["outcome"].eq("FAILED")
].merge(nonrenewal_ids, on="renewal_id", how="inner")
display(failed_first.groupby(["attempt_group", "detail_code"]).agg(
    failed_event_rows=("event_id", "nunique"),
    opportunities_with_code=("renewal_id", "nunique"),
))

support_first = event_opportunities[
    event_opportunities["event_type"].eq("SUPPORT_CONTACT")
].merge(nonrenewal_ids, on="renewal_id", how="inner")
display(support_first.groupby(["attempt_group", "detail_code"]).agg(
    support_event_rows=("event_id", "nunique"),
    opportunities_with_topic=("renewal_id", "nunique"),
))


                       opportunities  mean_prior_uses  mean_recent_uses  support_contact_share
attempt_group                                                                                 
Attempted, no success            540            8.428             8.417                  0.817
No recorded attempt             1136            6.208             1.092                  0.320
                                             failed_event_rows  opportunities_with_code
attempt_group         detail_code                                                      
Attempted, no success AUTHENTICATION_FAILED                327                      273
                      INSUFFICIENT_FUNDS                   149                      139
                      PSP_TIMEOUT                         1144                      522
                                                 support_event_rows  opportunities_with_topic
attempt_group         detail_code                                                     

## Trade-offs
Who might a discount help? Who needs a different repair? Which expected costs and intervention effects are not available in these files?

### Findings and recommendation

Who might benefit from a discount? Members who are genuinely price-sensitive are a reasonable test group, especially when there is a price query or a low-value signal. But the history cannot identify that group reliably or show the discount’s causal effect. In the mature data, all **1,060 discounted opportunities renewed**, versus 7,724 of 9,920 with no discount (77.9%). Since offer assignment was not randomized and the reasons for offers are absent, that 100% is an association and may reflect selection. Many members also renewed without a discount, so discounts are not necessary for every renewal.

Who needs a different repair? Members with failed attempts need a payment-recovery path (clear error messaging, retry or alternative payment method, and support escalation), not automatically a discount. Members with no attempt and falling benefit use may need a value/benefit reminder or simpler renewal journey; test price help separately rather than assume price is the reason.

The files contain renewal fees, offered discounts, and direct event costs, but not full annual contribution margin, full benefit servicing costs, payment-processing costs, intervention costs, or the counterfactual outcome without a discount. Annual renewal cash is not the same as accounting revenue or profit. Do not promise discount lift or lifetime value from these observational records.


In [23]:
discount_summary = mature_opportunities.copy()
discount_summary["stage"] = np.where(
    discount_summary["renewal_number"].eq(1), "First", "Later"
)
discount_summary = discount_summary.groupby(["stage", "offered_discount"]).agg(
    opportunities=("renewal_id", "nunique"),
    renewals=("renewed", "sum"),
).reset_index()
discount_summary["renewal_rate"] = (
    discount_summary["renewals"] / discount_summary["opportunities"]
)
display(discount_summary)


   stage  offered_discount  opportunities  renewals  renewal_rate
0  First                 0           5299      3623      0.683714
1  First               150            318       318      1.000000
2  First               250            163       163      1.000000
3  Later                 0           4621      4101      0.887470
4  Later               150            415       415      1.000000
5  Later               250            164       164      1.000000


## Signals
Propose a pilot outcome, leading signal, budget limit and stop/review rule. How would you establish additional renewals caused by the action?

### Suggested pilot

- **Primary outcome:** successful payment by the renewal deadline, analyzed by intention to treat (all assigned members remain in their assigned group).
- **Leading signals:** delivered renewal reminder, payment attempt after contact, successful retry, failure-code resolution, and support contacts. Add contact clicks/checkout starts if those can be instrumented; they are not in the current extract.
- **Budget limit:** Finance should set the maximum affordable cost per incremental renewal from conservative expected contribution margin. Set the total pilot cap before launch as eligible treatment count × that per-member cap, including discounts, outreach, support, and tooling. The present files do not supply enough cost data to set a responsible rupee amount.
- **Stop/review:** pause immediately for an erroneous charge, eligibility/privacy issue, or spend-cap breach. Review delivery and safety at a pre-set midpoint; judge effectiveness only after the planned sample’s renewal windows mature. Do not stop early just because a small interim rate looks favorable.
- **Causal estimate:** randomly assign eligible first-renewal opportunities to treatment and usual-care holdout, stratifying by attempted/no-attempt status and plan. Test payment recovery for failed attempts separately from value/renewal help for no-attempt cases; randomize any discount within a clearly defined eligible segment. Compare mature renewal rates between assigned groups and report confidence intervals and net contribution.

Choose sample size using the current baseline and a pre-agreed minimum lift worth acting on. Historical associations in this dataset are not estimates of pilot effect.


In [26]:
pilot_outcomes = pd.DataFrame([
    {"Measure": "Primary", "Definition": "Share paying successfully by renewal deadline; treatment vs. holdout (intention to treat)"},
    {"Measure": "Leading", "Definition": "Delivered contact, payment attempt after contact, retry success, failure resolution"},
    {"Measure": "Economic guardrail", "Definition": "Cost per incremental renewal <= Finance-approved expected contribution margin"},
    {"Measure": "Budget", "Definition": "Pre-set eligible treatment count x affordable per-member cost cap"},
    {"Measure": "Stop/review", "Definition": "Pause for charge/privacy/eligibility harm or cap breach; evaluate lift after windows mature"},
    {"Measure": "Design", "Definition": "Randomized holdout; stratify by renewal stage, attempt status, and plan; test interventions separately"},
])
display(pilot_outcomes)


              Measure                                                                                              Definition
0             Primary               Share paying successfully by renewal deadline; treatment vs. holdout (intention to treat)
1             Leading                     Delivered contact, payment attempt after contact, retry success, failure resolution
2  Economic guardrail                           Cost per incremental renewal <= Finance-approved expected contribution margin
3              Budget                                       Pre-set eligible treatment count x affordable per-member cost cap
4         Stop/review             Pause for charge/privacy/eligibility harm or cap breach; evaluate lift after windows mature
5              Design  Randomized holdout; stratify by renewal stage, attempt status, and plan; test interventions separately
